# 08 - Detector supervisado de deslizamiento mediante CNN 1D

El presente cuaderno documenta el entrenamiento de una rama algorítmica específica basada en aprendizaje profundo para la detección de la anomalía sutil de patinaje y frenazo, empleando las capturas de datos manuales `real_slip_manual_001` y `real_slip_manual_002`.

El modelo VAE se preserva como sistema de detección no supervisado para anomalías generales y de impacto. En contraste, esta rama supervisada se entrena de forma exclusiva para diferenciar los eventos `manual_slip_core` de los periodos `manual_normal_between`, excluyendo sistemáticamente cualquier evento categorizado bajo el prefijo `ignore_*`.

El procedimiento de evaluación principal se fundamenta en la validación cruzada:

- Entrenamiento sobre el conjunto `manual_001` y posterior evaluación sobre `manual_002`.
- Entrenamiento sobre el conjunto `manual_002` y posterior evaluación sobre `manual_001`.

Esta metodología previene el sesgo de evaluación asociado al uso de ventanas temporales cuasi-duplicadas provenientes de una misma serie de captura.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "app").is_dir() and (candidate / "tools").is_dir():
            return candidate
    fallback = Path("/workspace/TFM/inference_server")
    if (fallback / "app").is_dir():
        return fallback
    raise RuntimeError("No se ha encontrado el directorio raíz de inference_server.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
TFM_ROOT = REPO_ROOT.parent
print(f"REPO_ROOT={REPO_ROOT}")
print(f"TFM_ROOT={TFM_ROOT}")


## Configuración

Se establece el uso de la plataforma `wandb` por defecto, dado que resulta imperativo el registro sistemático de los resultados generados por esta rama predictiva. En caso de realizar ensayos en un entorno local sin conectividad a la red, se debe modificar la variable a `USE_WANDB = False`.


In [ ]:
MODEL_NAME = "slip_cnn_supervised_w50_s10_v1"
EPOCHS = 35
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.15
THRESHOLD = 0.5
USE_WANDB = True

WANDB_PROJECT = "tfm-railway-anomaly"
WANDB_ENTITY = None
WANDB_GROUP = "supervised-slip"

OUTPUT_DIR = TFM_ROOT / "datos" / "analisis" / "slip_manual" / "supervised_cnn"
print(OUTPUT_DIR)


## Verificación de datos manuales

En este apartado se comprueba la existencia de las etiquetas manuales y se certifica que las clases de interés disponen de un volumen suficiente de ventanas temporales para el modelado.


In [ ]:
label_files = {
    "001": TFM_ROOT / "datos" / "etiquetas" / "real_slip_manual_001_windows.csv",
    "002": TFM_ROOT / "datos" / "etiquetas" / "real_slip_manual_002_windows.csv",
}

for run_id, path in label_files.items():
    df = pd.read_csv(path)
    print(run_id, path)
    display(df["label"].value_counts().to_frame("windows"))


## Entrenamiento de la CNN supervisada

El núcleo de la lógica de entrenamiento se ha desacoplado en el script `herramientas/train_supervised_slip_cnn.py`. Este diseño arquitectónico permite su futura integración en el servicio *worker* con total independencia del entorno interactivo del cuaderno.


In [ ]:
command = [
    sys.executable,
    str(REPO_ROOT / "tools" / "train_supervised_slip_cnn.py"),
    "--model-name", MODEL_NAME,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY),
    "--dropout", str(DROPOUT),
    "--threshold", str(THRESHOLD),
    "--wandb-project", WANDB_PROJECT,
    "--wandb-entity", WANDB_ENTITY,
    "--wandb-group", WANDB_GROUP,
]
if USE_WANDB:
    command.append("--wandb")

print(" ".join(command))
completed = subprocess.run(command, cwd=REPO_ROOT, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"El proceso de entrenamiento ha fallado con el código de salida {completed.returncode}")


## Resultados de validación cruzada

La tabla principal expone la comparativa bidireccional del proceso de evaluación. Se documenta tanto el umbral operativo predeterminado de `0.5` como el umbral óptimo empírico observado durante la fase de prueba, este último con fines puramente diagnósticos. En un entorno de producción, no se implementará de forma automática el umbral óptimo sin un análisis previo y exhaustivo de la tasa de falsos positivos.


In [ ]:
summary_csv = OUTPUT_DIR / f"{MODEL_NAME}_crossrun_summary.csv"
results = pd.read_csv(summary_csv)
display(results[[
    "train_run", "test_run", "test_windows", "test_positives",
    "default_precision", "default_recall", "default_f1",
    "default_false_positive", "default_false_negative",
    "best_threshold", "best_precision", "best_recall", "best_f1",
    "best_false_positive", "best_false_negative",
]])


## Visualización de métricas: Precisión, Exhaustividad (Recall) y Puntuación F1

La siguiente representación gráfica se considera idónea para su inclusión en la memoria final del proyecto. Ilustra de manera empírica la capacidad de la rama supervisada para modelar y detectar la anomalía de deslizamiento (slip), validada de forma robusta mediante validación cruzada entre series temporales independientes.


In [ ]:
plot_df = results.copy()
plot_df["fold"] = "train " + plot_df["train_run"].astype(str) + " -> test " + plot_df["test_run"].astype(str)
metrics = ["default_precision", "default_recall", "default_f1"]
ax = plot_df.set_index("fold")[metrics].plot(kind="bar", figsize=(10, 4), ylim=(0, 1), rot=0)
ax.set_ylabel("Métrica")
ax.set_title("Evaluación cruzada: CNN supervisada para detección de deslizamiento")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Artefactos generados

El script de entrenamiento persiste un modelo individual para cada partición (fold) de validación. Para el despliegue final, la selección del modelo, la partición correspondiente y el umbral de decisión definitivo se llevarán a cabo tras un escrutinio riguroso del comportamiento de los falsos positivos.


In [ ]:
models = sorted((REPO_ROOT / "models").glob(f"{MODEL_NAME}_*.pth"))
for path in models:
    print(path.relative_to(REPO_ROOT), path.stat().st_size, "bytes")
print("summary:", summary_csv)


## Interpretación de resultados preliminares

Si la red convolucional supervisada logra sostener una métrica F1 elevada en ambas direcciones de la evaluación cruzada, se concluye que la arquitectura definitiva del sistema deberá configurarse con una topología de doble rama paralela:

- VAE: Orientado a la detección de anomalías generales y eventos mecánicos de fuerte impacto.
- CNN supervisada: Especializada en la identificación de anomalías sutiles de deslizamiento (slip) y patinaje.

Es importante señalar que la integración en un entorno de inferencia en tiempo real no demanda el aislamiento en procesos separados. El proceso en segundo plano (worker) posee la capacidad de instanciar ambos modelos en la misma zona de memoria y ejecutar sus correspondientes inferencias de forma concurrente sobre *buffers* de ventana temporal independientes. El sobrecoste computacional asociado se estima marginal, dada la reducida dimensionalidad paramétrica de la CNN y la brevedad de su ventana de análisis (0,5 s).
